# Notebook 02 — Bronze to Silver: Cleaning & Transformation

**Goal:** Apply cleaning rules to the raw bronze data and write a curated, 
analytics-ready dataset to the silver layer in Delta Lake format.

**Cleaning rules applied:**
- Filter trips outside 2023 (data quality issue from source)
- Remove records with null pickup/dropoff timestamps
- Remove trips with negative or zero fare
- Remove trips with zero or unrealistic distance (>100 miles)
- Remove trips with zero passengers
- Cap outliers on total_amount (>$500 likely data errors)
- Compute derived features: trip duration, hour of day, day of week
- Join with Taxi Zone lookup for human-readable location names

In [0]:
# ─────────────────────────────────────────────────────────────
# Setup: paths and imports
# ─────────────────────────────────────────────────────────────

from pyspark.sql.functions import (
    col, year, month, dayofweek, hour, when,
    unix_timestamp, round as spark_round, dayofmonth, date_format
)
from functools import reduce

storage_account_name = "staxidatakc250253104"
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"

print("✓ Paths configured")
print(f"  Bronze: {bronze_path}")
print(f"  Silver: {silver_path}")

✓ Paths configured
  Bronze: abfss://bronze@staxidatakc250253104.dfs.core.windows.net
  Silver: abfss://silver@staxidatakc250253104.dfs.core.windows.net


In [0]:
# Reload bronze data with schema normalisation 

problem_cols = ["airport_fee", "passenger_count", "RatecodeID",
                "PULocationID", "DOLocationID", "VendorID", "payment_type"]

dfs = []
for m in range(1, 13):
    mm = f"{m:02d}"
    df_month = spark.read.parquet(
        f"{bronze_path}/yellow_taxi/2023/yellow_tripdata_2023-{mm}.parquet"
    )
    for c in problem_cols:
        if c in df_month.columns:
            df_month = df_month.withColumn(c, col(c).cast("double"))
    dfs.append(df_month)

df_bronze = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

bronze_count = df_bronze.count()
print(f"✓ Bronze loaded: {bronze_count:,} records")

✓ Bronze loaded: 38,310,226 records


In [0]:
# Apply all cleaning rules in a single chained transformation

df_silver = (
    df_bronze
    # ─── Filter date range ──────────────────────────────────
    .filter(year("tpep_pickup_datetime") == 2023)
    
    # ─── Remove nulls in critical fields ────────────────────
    .filter(col("tpep_pickup_datetime").isNotNull())
    .filter(col("tpep_dropoff_datetime").isNotNull())
    .filter(col("PULocationID").isNotNull())
    .filter(col("DOLocationID").isNotNull())
    
    # ─── Sensible numeric ranges ────────────────────────────
    .filter(col("fare_amount") > 0)
    .filter(col("total_amount") > 0)
    .filter(col("total_amount") < 500)             
    .filter(col("trip_distance") > 0)
    .filter(col("trip_distance") < 100)            
    .filter(col("passenger_count") > 0)
    .filter(col("passenger_count") <= 6)           
    
    # ─── Compute derived features ───────────────────────────
    .withColumn("trip_duration_min",
                spark_round(
                    (unix_timestamp("tpep_dropoff_datetime") -
                     unix_timestamp("tpep_pickup_datetime")) / 60.0, 2))
    .filter(col("trip_duration_min") > 0)
    .filter(col("trip_duration_min") < 240)        
    
    .withColumn("pickup_hour", hour("tpep_pickup_datetime"))
    .withColumn("pickup_day_of_week", dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_day_name", date_format("tpep_pickup_datetime", "EEEE"))
    .withColumn("pickup_month_num", month("tpep_pickup_datetime"))
    .withColumn("pickup_date", col("tpep_pickup_datetime").cast("date"))
    
    # ─── Tip percentage (useful analytical feature) ─────────
    .withColumn("tip_pct",
                spark_round((col("tip_amount") / col("fare_amount")) * 100, 2))
    
    # ─── Time-of-day bucket ─────────────────────────────────
    .withColumn("time_of_day",
                when((col("pickup_hour") >= 6) & (col("pickup_hour") < 12), "Morning")
                .when((col("pickup_hour") >= 12) & (col("pickup_hour") < 17), "Afternoon")
                .when((col("pickup_hour") >= 17) & (col("pickup_hour") < 21), "Evening")
                .otherwise("Night"))
    
    # ─── Trip distance bucket ───────────────────────────────
    .withColumn("trip_distance_bucket",
                when(col("trip_distance") < 1, "Short (<1 mi)")
                .when(col("trip_distance") < 3, "Medium (1-3 mi)")
                .when(col("trip_distance") < 10, "Long (3-10 mi)")
                .otherwise("Very Long (10+ mi)"))
)

print("✓ Cleaning pipeline defined (lazy evaluation — not yet executed)")
print(f"  Output columns: {len(df_silver.columns)}")

✓ Cleaning pipeline defined (lazy evaluation — not yet executed)
  Output columns: 28


In [0]:
# Trigger execution and compare before/after
silver_count = df_silver.count()
removed = bronze_count - silver_count
removed_pct = (removed / bronze_count) * 100

print("CLEANING IMPACT")
print("=" * 50)
print(f"  Bronze records:  {bronze_count:>12,}")
print(f"  Silver records:  {silver_count:>12,}")
print(f"  Removed:         {removed:>12,}  ({removed_pct:.2f}%)")

CLEANING IMPACT
  Bronze records:    38,310,226
  Silver records:    35,575,601
  Removed:            2,734,625  (7.14%)


In [0]:
# Preview the cleaned, enriched dataset
display(df_silver.select(
    "tpep_pickup_datetime", "tpep_dropoff_datetime", 
    "trip_duration_min", "trip_distance", "trip_distance_bucket",
    "passenger_count", "PULocationID", "DOLocationID",
    "fare_amount", "tip_amount", "tip_pct", "total_amount",
    "pickup_hour", "time_of_day", "pickup_day_name"
).limit(15))

tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_min,trip_distance,trip_distance_bucket,passenger_count,PULocationID,DOLocationID,fare_amount,tip_amount,tip_pct,total_amount,pickup_hour,time_of_day,pickup_day_name
2023-01-01T00:32:10.000,2023-01-01T00:40:36.000,8.43,0.97,Short (<1 mi),1.0,161.0,141.0,9.3,0.0,0.0,14.3,0,Night,Sunday
2023-01-01T00:55:08.000,2023-01-01T01:01:27.000,6.32,1.1,Medium (1-3 mi),1.0,43.0,237.0,7.9,4.0,50.63,16.9,0,Night,Sunday
2023-01-01T00:25:04.000,2023-01-01T00:37:49.000,12.75,2.51,Medium (1-3 mi),1.0,48.0,238.0,14.9,15.0,100.67,34.9,0,Night,Sunday
2023-01-01T00:10:29.000,2023-01-01T00:21:19.000,10.83,1.43,Medium (1-3 mi),1.0,107.0,79.0,11.4,3.28,28.77,19.68,0,Night,Sunday
2023-01-01T00:50:34.000,2023-01-01T01:02:52.000,12.3,1.84,Medium (1-3 mi),1.0,161.0,137.0,12.8,10.0,78.13,27.8,0,Night,Sunday
2023-01-01T00:09:22.000,2023-01-01T00:19:49.000,10.45,1.66,Medium (1-3 mi),1.0,239.0,143.0,12.1,3.42,28.26,20.52,0,Night,Sunday
2023-01-01T00:27:12.000,2023-01-01T00:49:56.000,22.73,11.7,Very Long (10+ mi),1.0,142.0,200.0,45.7,10.74,23.5,64.44,0,Night,Sunday
2023-01-01T00:21:44.000,2023-01-01T00:36:40.000,14.93,2.95,Medium (1-3 mi),1.0,164.0,236.0,17.7,5.68,32.09,28.38,0,Night,Sunday
2023-01-01T00:39:42.000,2023-01-01T00:50:36.000,10.9,3.01,Long (3-10 mi),1.0,141.0,107.0,14.9,0.0,0.0,19.9,0,Night,Sunday
2023-01-01T00:53:01.000,2023-01-01T01:01:45.000,8.73,1.8,Medium (1-3 mi),1.0,234.0,68.0,11.4,3.28,28.77,19.68,0,Night,Sunday


In [0]:
# Write silver data as Delta Lake format, partitioned by month
# Delta gives us ACID transactions, time travel, and better query performance

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("pickup_month_num")
    .option("overwriteSchema", "true")
    .save(f"{silver_path}/yellow_taxi_2023")
)

print("✓ Silver layer written successfully")
print(f"  Location: {silver_path}/yellow_taxi_2023")
print(f"  Format: Delta Lake")
print(f"  Partitioned by: pickup_month_num")

✓ Silver layer written successfully
  Location: abfss://silver@staxidatakc250253104.dfs.core.windows.net/yellow_taxi_2023
  Format: Delta Lake
  Partitioned by: pickup_month_num


In [0]:
# Read back from silver to confirm the write worked
df_silver_verify = spark.read.format("delta").load(f"{silver_path}/yellow_taxi_2023")

print(f"✓ Silver layer verified")
print(f"  Record count: {df_silver_verify.count():,}")
print(f"  Column count: {len(df_silver_verify.columns)}")
print(f"\nColumns: {df_silver_verify.columns}")

✓ Silver layer verified
  Record count: 35,575,601
  Column count: 28

Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'trip_duration_min', 'pickup_hour', 'pickup_day_of_week', 'pickup_day_name', 'pickup_month_num', 'pickup_date', 'tip_pct', 'time_of_day', 'trip_distance_bucket']


In [0]:
# Sanity checks on the cleaned data
print("SILVER LAYER VALIDATION")
print("=" * 60)
print(f"\nNumeric column ranges:")
df_silver_verify.select(
    "fare_amount", "trip_distance", "trip_duration_min", "tip_amount", "total_amount"
).describe().display()

SILVER LAYER VALIDATION

Numeric column ranges:


summary,fare_amount,trip_distance,trip_duration_min,tip_amount,total_amount
count,35575601,35575601,35575601,35575601,35575601
mean,19.7190377984054,3.5076050926023834,16.40535913419948,3.598039510563956,28.89904459289134
stddev,17.93110706540624,4.56533327412668,13.239663811421012,4.031531269302549,22.65554939591618
min,0.01,0.01,0.02,0.0,0.01
max,497.0,99.92,239.95,444.42,499.95
